<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/GPAW_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Table of Contents

1. Install [GPAW](https://wiki.fysik.dtu.dk/gpaw/) and [JARVIS-Tools](https://github.com/usnistgov/jarvis)
2. Aluminum crystal structure, property calculations and visualizations
3. Silicon bandstructure
4. Example JARVIS-Leaderboard contribution
5. Interface energy calculation
Author: Kamal Choudhary (kamal.choudhary@nist.gov)


There are many DFT sotware. In this tutorial we will use GPAW. GPAW is a density-functional theory (DFT) Python code based on the projector-augmented wave (PAW) method and the [atomic simulation environment (ASE)](https://wiki.fysik.dtu.dk/ase/).

# 1. Installation

In [ ]:
%%time
!apt install python3-mpi4py cython3 libxc-dev gpaw-data
!pip -q install gpaw jarvis-tools spglib

In [ ]:
!gpaw test

In [ ]:
!gpaw info

# 2. FCC Aluminum

There are several ways to represent atomic structures such as CIF, XYZ, PDB, POSCAR formats etc. Usually, they have lattice matrix, chemical element and their crystallographic 3D coordinates information. Here we will use the POSCAR format.

In [ ]:
poscar = """MySystem
1.0
0.0 2.025 2.025
2.025 0.0 2.025
2.025 2.025 0.0
Al
1
direct
0.0 0.0 0.0 Al
"""
# Note difference between unit cell and conventional cell

In [ ]:
from jarvis.io.vasp.inputs import Poscar
atoms = Poscar.from_string(poscar).atoms #Poscar.from_file('POSCAR')


These atomic structures could be in primitive (least number of atoms), or conventionals (usually more number of atoms) forms, e.g. 1 vs 4 below. Lesser number of atoms reduces computational cost.

In [ ]:
# pip install -q spglib

In [ ]:
atoms.get_primitive_atoms

In [ ]:
atoms.get_conventional_atoms

In [ ]:
aluminum=atoms.ase_converter()

In [ ]:
from ase.visualize import view
view(atoms.make_supercell_matrix([3,3,3]).ase_converter(),viewer='x3d')

Lattice constants (for repeat units) are spcific to crystals. We can get reliable estimates of lattice constants using GPAW by relaxing the simulation cell.

In [ ]:
%%time
from gpaw import GPAW, PW
from ase.optimize import BFGS
from ase.constraints import UnitCellFilter
from jarvis.core.atoms import ase_to_atoms
# Set up bulk aluminum
print('Init',ase_to_atoms(aluminum))
# GPAW calculator setup
calc = GPAW(mode=PW(500),  # Plane wave cutoff
            xc='PBE',      # Exchange-correlation functional
            kpts=(6, 6, 6), # k-points
            convergence={'energy': 1e-6, 'density': 1e-4, 'eigenstates': 1e-8},
            txt='aluminum_gpaw.txt')

aluminum.set_calculator(calc)

# Relax the structure
uf = UnitCellFilter(aluminum)
opt = BFGS(uf)
opt.run(fmax=0.01)

print('Final',ase_to_atoms(aluminum))


Output: Energy, Forces, Stress (basic properties)

In [ ]:
aluminum.get_total_energy() #eV

In [ ]:
aluminum.get_forces() #eV/Angstrom

In [ ]:
# The stress on the cell should be as low as possible
initial_stress = aluminum.get_stress()
print('Stress 1x6 vector eV/Å³', initial_stress)
# Note stress can be 1x6 vector called Voigt notation or 3x3 matrix representation
# Here we are using Voigt notation

Note we can use energy,forces, stress information to develop/train force-fields which are used in molecular dynamics simulations.

Electronic density of states

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
energies, dos = calc.get_dos(spin=0, npts=2001, width=0.2)
e_f = calc.get_fermi_level()
plt.plot(energies - e_f, dos)
plt.xlim([-15, 10])
plt.ylim([0, 0.5])
plt.xlabel('Energy (eV)')
plt.ylabel('DOS (states/eV)')
plt.fill_between(energies - e_f, 0, dos, where=(energies - e_f < 0), facecolor='blue', alpha=0.25)
plt.axvline(x=0,linestyle='-.',color='green')
# The left hand side represent electron filled states
# The Fermi level (green line) is set to zero

Electronic band structure

In [ ]:
# Restart from ground state and fix potential:
calc2 = calc.fixed_density(
    nbands=16,
    symmetry='off',
    kpts={'path': 'GXWKL', 'npoints': 60},
    convergence={'bands': 8})

In [ ]:
%matplotlib inline
bs = calc2.band_structure()
bs.plot(filename='bandstructure.png', show=False, emax=25.0)

Electron density cloud

In [ ]:
density = calc.get_pseudo_density()
print(density.shape,density.max())

In [ ]:
import plotly.graph_objects as go
import numpy as np

# Create a sample NxNxN data
N = 15
x, y, z = np.mgrid[:N, :N, :N]
values = density  # Example 3D data

# Create the volume plot
fig = go.Figure(data=go.Volume(
    x=x.flatten(),
    y=y.flatten(),
    z=z.flatten(),
    value=values.flatten(),
    # isomin=-1,  # Minimum value to visualize
    # isomax=1,   # Maximum value to visualize
    opacity=0.1,  # Opacity of the volume
    surface_count=17,  # Number of isosurfaces
    colorscale='RdBu'  # Colorscale
))

# Update layout for better visualization
fig.update_layout(
    scene=dict(
        xaxis_title='X axis',
        yaxis_title='Y axis',
        zaxis_title='Z axis'
    ),
    title="3D Volume Plot"
)

# Show the plot
fig.show()


Generate stress-strain curve

In [ ]:
%%time
# Note we are using very loose k-point mesh for the tutorial, for production might need more
# All results so far at 0 K temperature
# Let's vary strain from 0 to 0.5 at interval of 0.05
from tqdm import tqdm
import numpy as np
from ase import units
j_atoms = ase_to_atoms(aluminum)
strains = np.arange(0, 0.5, 0.05)
stresses = []
for i in tqdm(strains):
    stress=[[i,0,0],[0,0,0],[0,0,0]]
    atoms=j_atoms.strain_atoms(stress).ase_converter()
    atoms.calc=calc
    final_stress = atoms.get_stress()
    print(i,final_stress[0])
    # eV/Ang^3 to Giga Pascal
    stresses.append(final_stress[0]/units.GPa)




In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
plt.plot(strains,stresses,'-o')
plt.xlabel('Strain (unitless)')
plt.ylabel('Stress (GPa)')

Get C11 elastic constant by getting slope of the curve in elastic regime

In [ ]:
C11 = (stresses[1]-stresses[0])/(strains[1]-strains[0])
print( 'C11', C11, 'GPa')

Get bulk modulus using volumetrix expansion and contraction and fitting [Equation Of State](https://wiki.fysik.dtu.dk/ase/ase/eos.html) with BirchMurnaghan equation

In [ ]:
%%time
import numpy as np
from ase.eos import calculate_eos
from ase.units import kJ
dx=np.arange(-0.05, 0.05, 0.01)
eos = calculate_eos(aluminum)
v, e, B = eos.fit()

kv = B / kJ * 1.0e24
print('Kv (GPa)',kv)


Plot equation of states

In [ ]:
eos.plot()

Exercise: calculate electron density of states, band structure, C11, and bulk modulus for Silicon

Single point calculation

In [ ]:
%%time
# Sample calculation from https://wiki.fysik.dtu.dk/gpaw/tutorialsexercises/structureoptimization/aluminium/aluminium.html
from ase import Atoms
from ase.visualize import view
from gpaw import GPAW, PW

name = 'Al-fcc'
a = 4.05  # fcc lattice parameter
b = a / 2

bulk = Atoms('Al',
             cell=[[0, b, b],
                   [b, 0, b],
                   [b, b, 0]],
             pbc=True)

view(bulk)

k = 4
calc = GPAW(mode=PW(300),       # cutoff
            kpts=(k, k, k),     # k-points
            txt=name + '.txt')  # output file

bulk.calc = calc

energy = bulk.get_potential_energy()
calc.write(name + '.gpw')
print('Energy:', energy, 'eV')

Geometry optimization

In [ ]:
from ase import Atoms
from ase.build import bulk
from ase.io import write
from ase.units import GPa
from gpaw import GPAW, PW
from ase.optimize import BFGS
from ase.constraints import UnitCellFilter
from ase.eos import EquationOfState
from jarvis.core.atoms import ase_to_atoms
# Set up bulk aluminum
aluminum = bulk('Al', 'fcc', a=4.05)
print('Init',ase_to_atoms(aluminum))
# GPAW calculator setup
calc = GPAW(mode=PW(500),  # Plane wave cutoff
            xc='PBE',      # Exchange-correlation functional
            kpts=(6, 6, 6), # k-points
            convergence={'energy': 1e-6, 'density': 1e-4, 'eigenstates': 1e-8},
            txt='aluminum_gpaw.txt')

aluminum.set_calculator(calc)

# Relax the structure
uf = UnitCellFilter(aluminum)
opt = BFGS(uf)
opt.run(fmax=0.01)

print('Final',ase_to_atoms(aluminum))


In [ ]:
from jarvis.db.figshare import get_jid_data
from jarvis.core.atoms import Atoms
from jarvis.core.kpoints import Kpoints3D
jid = 'JVASP-816'
dat = get_jid_data(jid=jid,dataset='dft_3d')
atoms = Atoms.from_dict(dat['atoms'])
atoms = atoms.get_conventional_atoms
ase_atoms = (atoms.get_conventional_atoms).ase_converter()
kp = Kpoints3D().automatic_length_mesh(
    #lattice_mat=atoms.lattice_mat,
    #length=10
    lattice_mat=atoms.lattice_mat, length=dat["kpoint_length_unit"]
)
kpts = kp._kpoints[0]



In [ ]:
kpts

In [ ]:
ase_atoms

In [ ]:
%%time
# Sample calculation from https://wiki.fysik.dtu.dk/gpaw/tutorialsexercises/structureoptimization/aluminium/aluminium.html
from ase import Atoms
from ase.visualize import view
from gpaw import GPAW, PW



calc = GPAW(mode=PW(500),       # cutoff
            kpts=kpts,     # k-points
            txt=name + '.txt')  # output file

ase_atoms.calc = calc

energy = ase_atoms.get_potential_energy()
#calc.write(name + '.gpw')
print('Energy:', energy, 'eV')

In [ ]:
%%time
import numpy as np
from ase.eos import calculate_eos
dx=np.arange(-0.05, 0.05, 0.01)
eos = calculate_eos(ase_atoms)
v, e, B = eos.fit()


In [ ]:
from ase.units import kJ
kv = B / kJ * 1.0e24
kv

# 3. Calculation of electronic band structure for Silicon

In [ ]:
from ase.build import bulk
from gpaw import GPAW, PW, FermiDirac

# Perform standard ground state calculation (with plane wave basis)
si = bulk('Si', 'diamond', 5.43)
calc = GPAW(mode=PW(200),
            xc='PBE',
            kpts=(8, 8, 8),
            random=True,  # random guess (needed if many empty bands required)
            occupations=FermiDirac(0.01),
            txt='Si_gs.txt')
si.calc = calc
si.get_potential_energy()
ef = calc.get_fermi_level()
calc.write('Si_gs.gpw')

In [ ]:
# Restart from ground state and fix potential:
calc = GPAW('Si_gs.gpw').fixed_density(
    nbands=16,
    symmetry='off',
    kpts={'path': 'GXWKL', 'npoints': 60},
    convergence={'bands': 8})

In [ ]:
%matplotlib inline
bs = calc.band_structure()
bs.plot(filename='bandstructure.png', show=False, emax=10.0)

#4. JARVIS-Leaderboard contribution

In [ ]:
%%time
# Source: https://github.com/usnistgov/jarvis_leaderboard/blob/main/jarvis_leaderboard/contributions/gpaw_lda/run.py
import pandas as pd
from gpaw import GPAW, PW, FermiDirac
from jarvis.db.figshare import get_jid_data
from jarvis.core.atoms import Atoms
from jarvis.core.kpoints import Kpoints3D
import time
from ase.build import bulk
from gpaw import GPAW, PW, FermiDirac
#source: https://wiki.fysik.dtu.dk/gpaw/tutorialsexercises/electronic/band_gap/band_gap.html


def get_band_gap(atoms=None, cutoff=500,kpts=[7,7,7],xc='LDA'):
    calc = GPAW(mode=PW(cutoff),
                xc=xc,
                kpts=kpts,  # Choose and converge carefully!
                occupations=FermiDirac(0.01),
                txt='gs.out')
    atoms.calc = calc
    atoms.get_potential_energy()
    # Calculate the discontinuity potential and the discontinuity
    homo, lumo = calc.get_homo_lumo()
    efermi = calc.get_fermi_level()
    bandgap=lumo-homo
    # response = calc.hamiltonian.xc.response
    #dxc_pot = response.calculate_discontinuity_potential(homo, lumo)
    # KS_gap, dxc = response.calculate_discontinuity(dxc_pot)

    # Fundamental band gap = Kohn-Sham band gap + derivative discontinuity
    #QP_gap = KS_gap + dxc

    print('Band gap', bandgap)
    print('Fermi level', efermi)
    #print(f'Discontinuity from GLLB-sc: {dxc:.2f} eV')
    #print(f'Fundamental band gap:       {QP_gap:.2f} eV')
    return bandgap

dat = get_jid_data(jid='JVASP-1002',dataset='dft_3d')
silicon = Atoms.from_dict(dat['atoms'])
ase_atoms=silicon.ase_converter()
kp = Kpoints3D().automatic_length_mesh(

    lattice_mat=dat['atoms']['lattice_mat'],
    length=dat["kpoint_length_unit"],
)
kpts = kp._kpoints[0]
t1=time.time()
KS_gap = get_band_gap(atoms=ase_atoms,kpts=kpts)
t2=time.time()
print('Time', t2-t1)

In [ ]:
f=open('ES-SinglePropertyPrediction-bandgap_JVASP_1002_Si-dft_3d-test-mae.csv','w')
f.write('id,prediction\n')
line=jid+','+str(KS_gap)+'\n'
f.write(line)
f.close()

In [ ]:
!zip ES-SinglePropertyPrediction-bandgap_JVASP_1002_Si-dft_3d-test-mae.csv.zip ES-SinglePropertyPrediction-bandgap_JVASP_1002_Si-dft_3d-test-mae.csv

You can download the zip file by clicking on the folder icon on the left side of this notebook

# Now you can

1) [Fork](https://github.com/usnistgov/jarvis_leaderboard/fork) the jarvis-leaderboard GitHub repository

2) Add a new folder in the `jarvis_leaderboard/jarvis_leaderboard
/contributions/` folder, e.g., `gpaw_lda_test`

3) In the folder, add the `ES-SinglePropertyPrediction-bandgap_JVASP_1002_Si-dft_3d-test-mae.csv.zip` file, an example `metadata.json` file, e.g. [this](https://github.com/usnistgov/jarvis_leaderboard/blob/main/jarvis_leaderboard/contributions/gpaw_lda/metadata.json) one, add a `run.sh` file, e.g. this [one](https://github.com/usnistgov/jarvis_leaderboard/blob/main/jarvis_leaderboard/contributions/gpaw_lda/run.sh)

4) Make a pull request from your forked repo to the main jarvis-leaderboard

# 5. A quick calculation of Al(111)-Al2O3(001) interface energy

In [ ]:
import os
import numpy as np
from jarvis.analysis.interface.zur import make_interface
from jarvis.analysis.defects.surface import Surface
from gpaw import GPAW, PW

def jarvis_atoms_to_energy(atoms=None,kp=[2,2,1],xc='PBE',ecut=300):
       name='jarvis_db_gpaw'
       calc = GPAW(mode=PW(ecut),       # cutoff
            kpts=kp,     # k-points
            txt=name + '.txt')
       ase_atoms=atoms.ase_converter()
       ase_atoms.calc=calc
       atoms_en = ase_atoms.get_potential_energy()
       return atoms_en

def get_interface_energy(
    film_atoms=None,
    subs_atoms=None,
    film_index=[1, 1, 1],
    subs_index=[0, 0, 1],
    film_thickness=25,
    subs_thickness=25,
    model_path="",
    seperation=3.0,
    vacuum=8.0,
    max_area_ratio_tol=1.00,
    max_area=500,
    ltol=0.05,
    atol=1,
    apply_strain=False,
    from_conventional_structure=True,
):
    film_surf = Surface(
        film_atoms,
        indices=film_index,
        from_conventional_structure=from_conventional_structure,
        thickness=film_thickness,
        vacuum=vacuum,
    ).make_surface()
    subs_surf = Surface(
        subs_atoms,
        indices=subs_index,
        from_conventional_structure=from_conventional_structure,
        thickness=subs_thickness,
        vacuum=vacuum,
    ).make_surface()
    het = make_interface(
        film=film_surf,
        subs=subs_surf,
        seperation=seperation,
        vacuum=vacuum,
        max_area_ratio_tol=max_area_ratio_tol,
        max_area=max_area,
        ltol=ltol,
        atol=atol,
        apply_strain=apply_strain,
    )
    film_en = jarvis_atoms_to_energy(het["film_sl"])
    print('film_en', film_en)
    subs_en = jarvis_atoms_to_energy(het["subs_sl"])
    print ('subs_en', subs_en)
    intf_en = jarvis_atoms_to_energy(het["interface"])
    print('intf_en', intf_en)

    m = het["interface"].lattice.matrix
    area = np.linalg.norm(np.cross(m[0], m[1]))
    intf_energy = 16 * (intf_en - subs_en - film_en) / (area)  # J/m2
    het["interface_energy"] = intf_energy
    return het




It might take more than a hour

In [ ]:
from jarvis.db.figshare import get_jid_data
from jarvis.core.atoms import Atoms
atoms_al = Atoms.from_dict(
    get_jid_data(dataset="dft_3d", jid="JVASP-816")["atoms"]
)
atoms_al2o3 = Atoms.from_dict(
    get_jid_data(dataset="dft_3d", jid="JVASP-32")["atoms"]
)
intf = get_interface_energy(
    film_atoms=atoms_al, subs_atoms=atoms_al2o3, film_thickness=10, subs_thickness=10
)
print(intf["interface_energy"] )
